In [ ]:
from pathlib import Path

import jax
import matplotlib.pyplot as plt
from dynamical_systems.continuous import Lorenz63
from dynamical_systems.transforms import TransformedODE
from dynamics_discovery.data import TimeSeriesDataset
from dynamics_discovery.data.loaders import RandomSampleBatching, SegmentLoader
from dynamics_discovery.loss_functions import DySLIMLoss
from dynamics_discovery.models import NeuralODE


jax.config.update("jax_enable_x64", True)
plt.style.use("matplotlib_utils.styles.dash_gridded")

datadir = Path("../../data")

In [2]:
noise = 0.1
downsample = 40  # 2
dataset, transform = (
    TimeSeriesDataset(
        *TimeSeriesDataset.from_hdf5(datadir / "lorenz63_large.hdf5")[::100]
    )
    .downsample(downsample)
    # .split_along_time(2000)[0]
    .split_along_time(500)[0]
    .add_noise(noise)
    .standardize()
)

lorenz_scaled = TransformedODE(Lorenz63(), transform)

dataset.u.shape

(50, 250, 3)

In [3]:
loader = SegmentLoader(
    dataset,
    segment_length=2,
    batch_strategy=RandomSampleBatching(2048),
)

In [4]:
loader_state = loader.init()
batch, loader_state = loader.load_batch(loader_state)

In [5]:
jax.tree.map(lambda x: x.shape, batch)

((2048, 2), (2048, 2, 3))

In [6]:
model_nn = NeuralODE(dim=3, width=64, depth=3)

In [9]:
loss_fn = DySLIMLoss(lambda_1=1.0, lambda_2=100.0)

In [10]:
loss_fn(model_nn, batch, None)

(Array(1.42684937, dtype=float64),
 {'mse': Array(1.08009619, dtype=float64),
  'mmd_1': Array(0.00188285, dtype=float64),
  'mmd_2': Array(0.0034487, dtype=float64)})